# LP Sweep Results — Pixel Error vs Ensemble Std Dev

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr
import seaborn as sns

## Utility functions

In [ ]:
def standardize_scorer_level(df, new_scorer="standard_scorer"):
    """Normalizes the scorer level of a 3-level MultiIndex column DataFrame.

    Args:
        df: DataFrame with a 3-level MultiIndex column (scorer, bodypart, coord).
        new_scorer: Replacement value for the scorer level. Defaults to
            'standard_scorer'.

    Returns:
        DataFrame with the scorer level replaced by new_scorer.
    """
    df.columns = pd.MultiIndex.from_tuples(
        [(new_scorer, bodypart, coord) for _, bodypart, coord in df.columns],
        names=df.columns.names,
    )
    return df


def compute_ensemble_stddev(df_preds, scorer_name="standard_scorer"):
    """Computes per-frame, per-keypoint ensemble standard deviation across models.

    Args:
        df_preds: List of prediction DataFrames, one per model. Each must have a
            3-level MultiIndex column: (scorer, bodypart, coord).
        scorer_name: Scorer name to normalize all DataFrames to before computing
            statistics. Defaults to 'standard_scorer'.

    Returns:
        Array of shape (n_frames, n_keypoints) containing the mean std dev across
        x/y coordinates for each frame and keypoint.
    """
    preds = []
    for i, df in enumerate(df_preds):
        df = standardize_scorer_level(df, scorer_name)

        cols_to_keep = [
            col for col in df.columns
            if not col[2].endswith("likelihood") and "Unnamed" not in col[2]
        ]
        df = df[cols_to_keep]

        if df.isna().any().any():
            nan_indices = df[df.isna().any(axis=1)].index
            nan_columns = df.columns[df.isna().any()]
            print(f"Warning: NaN values in DataFrame {i} — indices: {nan_indices}, columns: {nan_columns}")

        arr_np = df.to_numpy()
        try:
            arr = arr_np.reshape(df.shape[0], -1, 2)
        except ValueError as e:
            print(f"Reshape error: {e}  shape={arr_np.shape}")
            raise

        preds.append(arr)

    preds = np.stack(preds, axis=-1)  # (n_frames, n_keypoints, 2, n_models)

    if np.isnan(preds).any():
        print(f"Warning: NaN values in stacked preds array at {np.argwhere(np.isnan(preds))}")
    else:
        print("No NaN values in preds array.")

    return np.std(preds, axis=-1).mean(axis=-1)  # (n_frames, n_keypoints)


def compute_percentiles(arr, std_vals, percentiles):
    """Finds ensemble std dev thresholds corresponding to given percentiles.

    Given n_points_dict values (counts of frames with ens-std above each
    std_val threshold), returns the std_val at which each requested percentage
    of frames is retained.

    Args:
        arr: Array of frame counts where arr[i] is the number of frames with
            ensemble std dev above std_vals[i].
        std_vals: Array of std dev threshold values, one per entry in arr.
        percentiles: List of percentages (0-100) to look up.

    Returns:
        Tuple of (vals, prctiles):
            vals: std dev threshold values for each requested percentile.
            prctiles: Achieved percentile for each threshold (may differ from
                requested if there is insufficient data).
    """
    num_pts = arr[0]
    vals, prctiles = [], []
    for p in percentiles:
        idx = np.argmin(np.abs(arr - num_pts * p / 100))
        achieved = arr[idx] / num_pts * 100 if idx == len(arr) - 1 else p
        vals.append(std_vals[idx])
        prctiles.append(np.round(achieved, 2))
    return vals, prctiles


class Ensemble:
    data_to_plot: dict[str, str]  # model label -> path to predictions_test.csv
    pred_csv_list: list[str]
    error_csv_list: list[str]
    df_pred_list: list[pd.DataFrame]
    df_error_list: list[pd.DataFrame]
    n_points_dict: dict[str, dict[str, int]]  # model label -> std_val_idx -> n_points
    std_vals: list[float]
    df_w_vars: pd.DataFrame
    df_line2: pd.DataFrame


def build_ensemble(ens: Ensemble, std_val_max: float = 8.0):
    """Loads model predictions and pixel errors, then computes ensemble statistics.

    Reads prediction CSVs and their corresponding pixel error CSVs for all
    models in ens.data_to_plot, computes ensemble std dev across models, and
    builds summary DataFrames binned by ensemble std dev threshold.

    The error CSV for each prediction CSV is derived by inserting '_pixel_error'
    before the filename suffix (e.g., predictions_test.csv ->
    predictions_pixel_error_test.csv).

    Args:
        ens: Ensemble instance with data_to_plot populated. Keys are model
            labels (e.g., 'resnet50_animal_ap10k.0'), values are absolute paths
            to predictions CSV files.
        std_val_max: Upper bound (exclusive) of the ensemble std dev range used
            for binning. Increase for datasets with high variance. Defaults to 8.0.

    Returns:
        The same Ensemble instance with the following attributes populated:
            pred_csv_list: Paths to prediction CSVs.
            error_csv_list: Paths to pixel error CSVs.
            df_pred_list: Loaded prediction DataFrames.
            df_error_list: Loaded pixel error DataFrames.
            n_points_dict: Per-model arrays of frame counts above each std threshold.
            std_vals: Array of std dev thresholds used for binning.
            df_w_vars: Long-form DataFrame with per-frame pixel errors and std devs.
            df_line2: Aggregated DataFrame binned by std dev threshold for plotting.
    """
    model_names_list = list(ens.data_to_plot.keys())
    pred_csv_list = [Path(v) for v in ens.data_to_plot.values()]
    error_csv_list = [
        p.with_stem(
            p.stem
            .replace("_new", "_pixel_error_new")
            .replace("_test", "_pixel_error_test")
        )
        for p in pred_csv_list
    ]

    df_pred_list, df_error_list = [], []
    for pred_csv, error_csv in zip(pred_csv_list, error_csv_list):
        df_pred_list.append(pd.read_csv(pred_csv, header=[0, 1, 2], index_col=0).sort_index())
        df = pd.read_csv(error_csv, header=[0], index_col=0).sort_index()
        if "set" in df.columns:
            df = df.drop(columns=["set"])
        df_error_list.append(df)

    ens_stddev = compute_ensemble_stddev(df_pred_list)

    df_w_vars = []
    for df_error, df_pred, model_name in zip(df_error_list, df_pred_list, model_names_list):
        assert (df_error.index == df_pred_list[0].index).all()
        assert (df_pred.index == df_pred_list[0].index).all()

        print(f"Total pixel error for model {model_name}: {df_error.sum().sum()}")

        for i, kp in enumerate(df_error.columns):
            index = [f"{frame}_{model_name}_{kp}" for frame in df_error.index]
            df_w_vars.append(pd.DataFrame(
                {
                    "pixel_error": df_error[kp].values,
                    "likelihood": df_pred.loc[:, ("standard_scorer", kp, "likelihood")].values,
                    "ens-std": ens_stddev[:, i],
                    "ens-std-prctile": [
                        np.sum(ens_stddev < p) / ens_stddev.size for p in ens_stddev[:, i]
                    ],
                    "ens-std-prctile-kp": [
                        np.sum(ens_stddev[:, i] < p) / ens_stddev[:, i].size for p in ens_stddev[:, i]
                    ],
                    "keypoint": kp,
                    "model": model_name,
                },
                index=index,
            ))

    df_w_vars = pd.concat(df_w_vars)
    std_vals = np.arange(0, std_val_max, 0.25)
    n_points_dict = {m: np.nan * np.zeros_like(std_vals) for m in model_names_list}

    df_line2 = []
    for s, std in enumerate(std_vals):
        df_tmp_ = df_w_vars[df_w_vars["ens-std"] > std]
        for model_name in model_names_list:
            d = df_tmp_[df_tmp_.model == model_name]
            n_points_dict[model_name][s] = np.sum(~d["pixel_error"].isna())
            df_line2.append(pd.DataFrame(
                {
                    "ens-std": std,
                    "model": model_name,
                    "pixel_error": d.pixel_error.to_numpy(),
                    "keypoint": d["keypoint"].to_numpy(),
                },
                index=[f"{row}_{s}" for row in d.index],
            ))

    ens.pred_csv_list = pred_csv_list
    ens.error_csv_list = error_csv_list
    ens.df_pred_list = df_pred_list
    ens.df_error_list = df_error_list
    ens.n_points_dict = n_points_dict
    ens.std_vals = std_vals
    ens.df_w_vars = df_w_vars
    ens.df_line2 = pd.concat(df_line2)
    return ens


## Configuration

Edit the variables in the cell below to select which sweep results to plot.

In [ ]:
# ============================================================
# User configuration — edit these variables
# ============================================================

base_dir = '/teamspace/lightning_storage/sweep-results'
dataset = 'mirror-mouse-fused'   # must match folder name under base_dir

backbones = [
    'resnet50_animal_ap10k',
    # 'vits_dino',
    # 'vitb_dino',
    # 'vits_dinov2',
    # 'vitb_dinov2',
    # 'vits_dinov3',
    # 'vitb_dinov3',
]

losses = 'supervised'   # 'supervised' | 'temporal' | 'pca_singleview+temporal' | ...
train_frames = 100
seeds = [0, 1, 2]

# y-axis limit for the main plot; add entries for new datasets as needed
ylim_map = {
    'mirror-mouse-fused': 75,
    'mirror-fish': 46,
    'crim13': 45,
}

# one color per backbone entry above, in the same order
colors = [
    '#000000',  # black
    '#ad3305',  # orange
    '#e05b28',  # light orange
    '#08961d',  # green
    '#32d14a',  # light green
    '#034e96',  # dark blue
    '#1E90FF',  # dodger blue
]

# ============================================================
# Build model paths from sweep directory structure
# ============================================================

def _sanitize(s):
    return s.replace('/', '_').replace('.', '_')

dataset_name = dataset  # referenced by build_ensemble and plotting cells

ens = Ensemble()
ens.data_to_plot = {}
for backbone in backbones:
    for i, seed in enumerate(seeds):
        path = (
            Path(base_dir) / dataset / _sanitize(backbone)
            / losses / f'tf{train_frames}' / f'seed{seed}'
            / 'predictions_test.csv'
        )
        ens.data_to_plot[f'{backbone}.{i}'] = str(path)

build_ensemble(ens)

models_to_plot = backbones
color_mapping = {m: c for m, c in zip(models_to_plot, colors)}

def major_model(s):
    return '.'.join(s.split('.')[:-1])
ens.df_line2['model2'] = ens.df_line2['model'].apply(major_model)


## Main plot

In [ ]:
df_line2 = ens.df_line2
n_points_dict = ens.n_points_dict
std_vals = ens.std_vals

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for model in models_to_plot:
    data = df_line2[df_line2['model2'] == model].reset_index(drop=True)
    sns.lineplot(x='ens-std', y='pixel_error', data=data, label=model,
                 color=color_mapping[model], ax=ax, errorbar='se', linewidth=2)

ax.set_title(f'{dataset_name}', fontsize=16)
ax.set_ylabel('Pixel error', fontsize=14)
ax.set_xlabel('Ensemble std dev', fontsize=14)
ax.set_ylim(bottom=0)
if dataset_name in ylim_map:
    ax.set_ylim(top=ylim_map[dataset_name])
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=12)

# percentile annotations
percentiles = [95, 50, 5]
vals, prctiles = compute_percentiles(
    arr=n_points_dict[models_to_plot[0] + '.0'],
    std_vals=std_vals,
    percentiles=percentiles,
)
for p, v in zip(prctiles, vals):
    ax.axvline(v, ymax=0.95, linestyle='--', linewidth=1, color='black', alpha=0.5)
    ax.text(v, ax.get_ylim()[1], f'{p}%', ha='left', va='top', fontsize=10, rotation=90)

plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()
